<a href="https://colab.research.google.com/github/Gsrichandana/Telecom_churn/blob/feature%2Fmodel_0.93026_accuracy/Telecom_Churn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

telecom_churn_case_study_hackathon_c_69_path = kagglehub.competition_download('telecom-churn-case-study-hackathon-c-69')

print('Data source import complete.')


# 0. Problem statement

In the telecom industry, customers are able to choose from multiple service providers and actively switch from one operator to another. In this highly competitive market, the telecommunications industry experiences an average of 15-25% annual churn rate. Given the fact that it costs 5-10 times more to acquire a new customer than to retain an existing one, customer retention has now become even more important than customer acquisition.

For many incumbent operators, retaining high profitable customers is the number one business
goal. To reduce customer churn, telecom companies need to predict which customers are at high risk of churn. In this project, you will analyze customer-level data of a leading telecom firm, build predictive models to identify customers at high risk of churn, and identify the main indicators of churn.

In this competition, your goal is *to build a machine learning model that is able to predict churning customers based on the features provided for their usage.*

**Customer behaviour during churn:**

Customers usually do not decide to switch to another competitor instantly, but rather over a
period of time (this is especially applicable to high-value customers). In churn prediction, we
assume that there are three phases of customer lifecycle :

1. <u>The ‘good’ phase:</u> In this phase, the customer is happy with the service and behaves as usual.

2. <u>The ‘action’ phase:</u> The customer experience starts to sore in this phase, for e.g. he/she gets a compelling offer from a competitor, faces unjust charges, becomes unhappy with service quality etc. In this phase, the customer usually shows different behaviour than the ‘good’ months. It is crucial to identify high-churn-risk customers in this phase, since some corrective actions can be taken at this point (such as matching the competitor’s offer/improving the service quality etc.)

3. <u>The ‘churn’ phase:</u> In this phase, the customer is said to have churned. In this case, since you are working over a four-month window, the first two months are the ‘good’ phase, the third month is the ‘action’ phase, while the fourth month (September) is the ‘churn’ phase.

# 1. Loading dependencies & datasets

Lets start by loading our dependencies. We can keep adding any imports to this cell block, as we write more and more code.

In [ ]:
pip install missingno xgboost

In [ ]:
#Data Structures
import pandas as pd
import numpy as np
import re
import os
import missingno as msno

#Sklearn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, precision_score, recall_score, classification_report
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer

#Plotting
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

#Others
import warnings
import xgboost as xgb
warnings.filterwarnings('ignore')

%matplotlib inline
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

Next, we load our datasets and the data dictionary file.

The **train.csv** file contains both dependent and independent features, while the **test.csv** contains only the independent variables.

So, for model selection, I will create our own train/test dataset from the **train.csv** and use the model to predict the solution using the features in unseen test.csv data for submission.

In [ ]:
#COMMENT THIS SECTION INCASE RUNNING THIS NOTEBOOK LOCALLY

#Checking the kaggle paths for the uploaded datasets
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
#INCASE RUNNING THIS LOCALLY, PASS THE RELATIVE PATH OF THE CSV FILES BELOW
#(e.g. if files are in same folder as notebook, simple write "train.csv" as path)

data = pd.read_csv("/kaggle/input/telecom-churn-case-study-hackathon-c-69/train.csv")
unseen = pd.read_csv("/kaggle/input/telecom-churn-case-study-hackathon-c-69/test.csv")
sample = pd.read_csv("/kaggle/input/telecom-churn-case-study-hackathon-c-69/sample.csv")
data_dict = pd.read_csv("/kaggle/input/telecom-churn-case-study-hackathon-c-69/data_dictionary.csv")

# data = pd.read_csv("train.csv")
# unseen = pd.read_csv("test.csv")
# sample = pd.read_csv("sample.csv")
# data_dict = pd.read_csv("data_dictionary.csv")

print(data.shape)
print(unseen.shape)
print(sample.shape)
print(data_dict.shape)

In [ ]:
data.head()

In [ ]:
data.info(verbose=1)

In [ ]:
data.describe(include="all")

# 2. Create X, y and then Train test split

Lets create X and y datasets and skip "circle_id" since it has only 1 unique value

In [ ]:
data['circle_id'].unique()

In [ ]:
X = data.drop(['circle_id'],axis=1).iloc[:,:-1]
y = data.iloc[:,-1]

X.shape, y.shape

Splitting train and test data to avoid any contamination of the test data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
X_train.head()

In [ ]:
#Dropping the id column
X_train.drop(['id'],axis=1,inplace=True)
X_test.drop(['id'],axis=1,inplace=True)

# 3. Handling Missing data

First lets analyse the missing data. We can use missingno library for quick visualizations.

In [ ]:
msno.bar(X_train)

In [ ]:
msno.matrix(X_train)

Lets also calculate the % missing data for each column:

In [ ]:
missing_data_percent = 100*X_train.isnull().sum()/len(y_train)
missing_data_percent

Since too much missing information would make a column not really a great predictor for churn, we drop these columns and keep only the ones which have less than 40% missing data.

In [ ]:
new_vars = missing_data_percent[missing_data_percent.le(40)].index
new_vars

In [ ]:
X_train_filtered = X_train[new_vars]
X_train_filtered.shape

In [ ]:
X_train_filtered.isnull().sum()

In [ ]:
X_train_filtered.dtypes

In [ ]:
pd.concat([X_train_filtered.isnull().sum(), X_train_filtered.dtypes], axis=1,
          keys=['Missing Values', 'Data Types'])

In [ ]:
data_dict

Filling the missing values of loc_og_t2o_mou, std_og_t2o_mou, loc_ic_t2o_mou with the average of that columns

In [ ]:
X_train_filtered['loc_og_t2o_mou'] = X_train_filtered['loc_og_t2o_mou'].fillna(X_train_filtered['loc_og_t2o_mou'].mean())
X_train_filtered['std_og_t2o_mou'] = X_train_filtered['std_og_t2o_mou'].fillna(X_train_filtered['std_og_t2o_mou'].mean())
X_train_filtered['loc_ic_t2o_mou'] = X_train_filtered['loc_ic_t2o_mou'].fillna(X_train_filtered['loc_ic_t2o_mou'].mean())

In [ ]:
pd.concat([X_train_filtered.isnull().sum(), X_train_filtered.dtypes], axis=1,
          keys=['Missing Values', 'Data Types'])

Converting the 3 columns last_date_of_month_6, last_date_of_month_7, last_date_of_month_8 into numeric by extracting only the day from that date and filling the missing values of last_date_of_month_7	and last_date_of_month_8 with 31 as that is the last day of those months

In [ ]:
X_train_filtered['last_date_of_month_6'] = 30
X_train_filtered['last_date_of_month_7'] = 31
X_train_filtered['last_date_of_month_8'] = 31
X_train_filtered['last_date_of_month_6'].astype('int64')
X_train_filtered['last_date_of_month_7'].astype('int64')
X_train_filtered['last_date_of_month_8'].astype('int64')
pd.concat([X_train_filtered.isnull().sum(), X_train_filtered.dtypes], axis=1,
          keys=['Missing Values', 'Data Types'])

Extracting only the day from the date columns: date_of_last_rech_6, date_of_last_rech_7, date_of_last_rech_8

In [ ]:
X_train_filtered['date_of_last_rech_6'] = pd.to_datetime(X_train_filtered['date_of_last_rech_6'])
X_train_filtered['date_of_last_rech_6'] = X_train_filtered['date_of_last_rech_6'].dt.day
X_train_filtered['date_of_last_rech_7'] = pd.to_datetime(X_train_filtered['date_of_last_rech_7'])
X_train_filtered['date_of_last_rech_7'] = X_train_filtered['date_of_last_rech_7'].dt.day
X_train_filtered['date_of_last_rech_8'] = pd.to_datetime(X_train_filtered['date_of_last_rech_8'])
X_train_filtered['date_of_last_rech_8'] = X_train_filtered['date_of_last_rech_8'].dt.day

Analysing the realtion between the missing dates and the churn probability and filling the missing values here with 0 if the churn value for that user is 1 or else filling it with the mode of that month

In [ ]:
#date_of_last_rech_6
X_train_filtered['date_of_last_rech_6'] = X_train_filtered['date_of_last_rech_6'].fillna(X_train_filtered['date_of_last_rech_6'].mode()[0])

In [ ]:
#date_of_last_rech_7
X_train_filtered['date_of_last_rech_7'] = X_train_filtered['date_of_last_rech_7'].fillna(X_train_filtered['date_of_last_rech_7'].mode()[0])

In [ ]:
#date_of_last_rech_8
X_train_filtered['date_of_last_rech_8'] = X_train_filtered['date_of_last_rech_8'].fillna(X_train_filtered['date_of_last_rech_8'].mode()[0])

In [ ]:
X_train_filtered['date_of_last_rech_6'] = X_train_filtered['date_of_last_rech_6'].astype('int64')
X_train_filtered['date_of_last_rech_7'] = X_train_filtered['date_of_last_rech_7'].astype('int64')
X_train_filtered['date_of_last_rech_8'] = X_train_filtered['date_of_last_rech_8'].astype('int64')
pd.concat([X_train_filtered.isnull().sum(), X_train_filtered.dtypes], axis=1,
          keys=['Missing Values', 'Data Types'])

The rest of the columns seem to have a pattern.
All the columns are having the same number of missing values in a monthly pattern.
It could mean that the person was inactive in that month since all of his data in that month is null.
So filling the missing values in these columns with 0 might be the best approach considering that the user was inactive in that month.

In [ ]:
X_train_filtered = X_train_filtered.fillna(0)
pd.concat([X_train_filtered.isnull().sum(), X_train_filtered.dtypes], axis=1,
          keys=['Missing Values', 'Data Types'])

In [ ]:
msno.bar(X_train_filtered)

In [ ]:
X_train_filtered.describe()

# 4. Exploratory Data Analysis & Preprocessing

Lets start by analysing the univariate distributions of each feature.

In [ ]:
plt.figure(figsize=(30,8))
plt.xticks(rotation=45)
sns.boxplot(data = X_train_filtered)

### 4.1 Handling outliers

The box plots of these features show there a lot of outliers. These can be capped with k-sigma method.

In [ ]:
def cap_outliers(array, k=2.5):
    upper_limit = array.mean() + k*array.std()
    lower_limit = array.mean() - k*array.std()
    array[array<lower_limit] = lower_limit
    array[array>upper_limit] = upper_limit
    return array

In [ ]:
X_train_filtered1 = X_train_filtered.apply(cap_outliers, axis=0)

plt.figure(figsize=(30,8))
plt.xticks(rotation=45)
sns.boxplot(data = X_train_filtered1)

### 4.2 Feature scaling

Lets also scale the features by scaling them with Standard scaler (few other alternates are min-max scaling and Z-scaling).

In [ ]:
scale = StandardScaler()
X_train_filtered2 = scale.fit_transform(X_train_filtered1)

In [ ]:
plt.figure(figsize=(30,8))
plt.xticks(rotation=45)
sns.boxplot(data = pd.DataFrame(X_train_filtered2, columns=X_train_filtered1.columns))

From this boxplot we can see that outlier treatment and scaling are not very affective on this data and some of them might be unnecessary features so I am going to do Feature Selection using Random Forest first and then handle outliers and do Scaling.

From this boxplot we can see that outlier treatment using k-sigma didn't work well so trying percentile based capping and scaling.

In [ ]:
#Percentile Capping
lower_limit = X_train_filtered.quantile(0.01, axis=0)
upper_limit = X_train_filtered.quantile(0.99, axis=0)

X_train_filtered1 = X_train_filtered.clip(lower=lower_limit, upper=upper_limit, axis=1)

plt.figure(figsize=(30,8))
plt.xticks(rotation=45)
sns.boxplot(data = X_train_filtered1)

In [ ]:
#Scaling
scale = StandardScaler()
X_train_filtered2 = scale.fit_transform(X_train_filtered1)

plt.figure(figsize=(30,8))
plt.xticks(rotation=45)
sns.boxplot(data = pd.DataFrame(X_train_filtered2, columns=X_train_filtered1.columns))

In [ ]:
X_train_filtered2 = pd.DataFrame(X_train_filtered2, columns=X_train_filtered1.columns)

In [ ]:
plt.figure(figsize=(40,30))
sns.heatmap(pd.DataFrame(X_train_filtered2, columns=new_vars).corr())

In [ ]:
#Distribution for the churn probability
sns.histplot(y_train)

# 5. Feature engineering and selection

Let's understand feature importances for raw features as well as components to decide top features for modelling.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
rf.fit(X_train_filtered2, y_train)

In [ ]:
feature_importances = pd.DataFrame({'col':new_vars, 'importance':rf.feature_importances_})

In [ ]:
plt.figure(figsize=(30,8))
plt.xticks(rotation=45)
plt.bar(feature_importances['col'], feature_importances['importance'])

Since there are too many features, I am going to use PCA to reduce the dimensionality

In [ ]:
# Set a threshold for feature importance (e.g., greater than 0.01)
threshold = 0.01
important_features = feature_importances[feature_importances['importance'] > threshold]['col']

# Apply PCA and decide the number of components (e.g., keep 95% variance)
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train_filtered2[important_features])

# Check explained variance ratio
print("Explained Variance Ratio:", pca.explained_variance_ratio_)
print("Total Variance Explained:", sum(pca.explained_variance_ratio_))

In [ ]:
sns.scatterplot(x=X_train_pca[:,0], y=X_train_pca[:,1], hue=y_train)

In [ ]:
sns.scatterplot(x=X_train_pca[:,1], y=X_train_pca[:,2], hue=y_train)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, n_jobs=-1)
rf.fit(X_train_pca, y_train)

feature_importances = pd.DataFrame({'col':['component_'+str(i) for i in range(14)],
                                    'importance':rf.feature_importances_})

plt.figure(figsize=(30,8))
plt.xticks(rotation=45)
plt.bar(feature_importances['col'], feature_importances['importance'])

# 6. Model building

Let's build a quick model with logistic regression and the first 2 PCA components.

In [ ]:
lr = LogisticRegression(max_iter=1000, tol=0.001, solver='sag')
lr.fit(X_train_pca[:,:2], y_train)

In [ ]:
lr.score(X_train_pca[:,:2], y_train)

In [ ]:
# # Make predictions
# y_pred_lr = lr.predict(X_test)

# # Get train accuracy
# train_accuracy = lr.score(X_train_pca[:,:2], y_train)
# print(f"Train Accuracy: {train_accuracy:.4f}")

# # Get test accuracy
# test_accuracy = lr.score(X_test, y_test)
# print(f"Test Accuracy: {test_accuracy:.4f}")

# # Evaluate model performance
# print("Classification Report:")
# print(classification_report(y_test, y_pred_lr))

# print("Confusion Matrix:")
# print(confusion_matrix(y_test, y_pred_lr))

# Building Pipeline

The steps of this pipeline would be the following -
1. Imputation - doing separately
2. Outlier Handling
3. Scaling
4. Feature Selection
5. PCA
6. Classification model


In [ ]:
# Custom imputation function
def custom_imputation(X, y=None):
    # Fill NaN values in specific columns with their means
    X['loc_og_t2o_mou'] = X['loc_og_t2o_mou'].fillna(X['loc_og_t2o_mou'].mean())
    X['std_og_t2o_mou'] = X['std_og_t2o_mou'].fillna(X['std_og_t2o_mou'].mean())
    X['loc_ic_t2o_mou'] = X['loc_ic_t2o_mou'].fillna(X['loc_ic_t2o_mou'].mean())

    # Set specific values for last_date_of_month columns
    X['last_date_of_month_6'] = 30
    X['last_date_of_month_7'] = 31
    X['last_date_of_month_8'] = 31

    # Convert to int64
    X['last_date_of_month_6'] = X['last_date_of_month_6'].astype('int64')
    X['last_date_of_month_7'] = X['last_date_of_month_7'].astype('int64')
    X['last_date_of_month_8'] = X['last_date_of_month_8'].astype('int64')

    # Convert date_of_last_rech columns to day of the month
    X['date_of_last_rech_6'] = pd.to_datetime(X['date_of_last_rech_6']).dt.day
    X['date_of_last_rech_7'] = pd.to_datetime(X['date_of_last_rech_7']).dt.day
    X['date_of_last_rech_8'] = pd.to_datetime(X['date_of_last_rech_8']).dt.day

    # Impute mode values based on y_train for the date columns
    X['date_of_last_rech_6'] = X['date_of_last_rech_6'].fillna(X['date_of_last_rech_6'].mode()[0])

    X['date_of_last_rech_7'] = X['date_of_last_rech_7'].fillna(X['date_of_last_rech_7'].mode()[0])

    X['date_of_last_rech_8'] = X['date_of_last_rech_8'].fillna(X['date_of_last_rech_8'].mode()[0])

    # Convert the date columns to int64
    X['date_of_last_rech_6'] = X['date_of_last_rech_6'].astype('int64')
    X['date_of_last_rech_7'] = X['date_of_last_rech_7'].astype('int64')
    X['date_of_last_rech_8'] = X['date_of_last_rech_8'].astype('int64')

    # Fill any remaining NaN values with 0
    X = X.fillna(0)

    return X

In [ ]:
def percentile_capping(X):
    # Compute the 1st and 99th percentiles for each column
    lower_limit = X.quantile(0.01, axis=0)
    upper_limit = X.quantile(0.99, axis=0)

    # Apply the capping to each column
    X_capped = X.clip(lower=lower_limit, upper=upper_limit, axis=1)

    return X_capped

In [ ]:
def scaling(X):
    #Scaling
    scale = StandardScaler()
    X_scaled = scale.fit_transform(X)
    X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

    return X_scaled

In [ ]:
class FeatureSelection(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.01):
        self.threshold = threshold
        self.important_features_ = None

    def fit(self, X, y=None):
        # Fit the RandomForestClassifier and calculate feature importances
        rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state = 75)
        rf.fit(X, y)

        # Get feature importances
        feature_importances = pd.DataFrame({
            'col': X.columns,
            'importance': rf.feature_importances_
        })

        # Select features that exceed the threshold
        self.important_features_ = feature_importances[feature_importances['importance'] > self.threshold]['col'].values
        return self

    def transform(self, X):
        # Select the important features
        return X[self.important_features_]

In [ ]:
#Using logistic regression
lr = LogisticRegression(max_iter=1000, tol=0.001, solver='sag')

In [ ]:
#Trying to handle class imbalance
logreg = LogisticRegression(max_iter=1000, tol=0.001, solver='sag',class_weight='balanced')

In [ ]:
#Hyperparameter tuning the Linear Regression model
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'saga', 'newton-cg', 'lbfgs'],
    'max_iter': [100, 500, 1000],
    'tol': [1e-4, 1e-3, 1e-2],
}
model = LogisticRegression()

# Setup GridSearchCV with 5-fold cross-validation
grid_search_lr = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, n_jobs=-1, verbose=1)

In [ ]:
#Random Forest with class weight
rf = RandomForestClassifier(n_estimators=50, max_depth=20, min_samples_split=2, min_samples_leaf=1, random_state=42, class_weight='balanced')

In [ ]:
#Hyperparameter tuning the Random Forest model
param_grid_rf = {
    'n_estimators': [10, 20, 30, 50],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['auto', 'sqrt', 'log2'],
    'bootstrap': [True, False],
    'oob_score': [True, False]
}

rf_model = RandomForestClassifier(random_state=42)

# Setup GridSearchCV with 5-fold cross-validation
grid_search_rf = GridSearchCV(estimator=rf_model, param_grid=param_grid_rf, cv=2, n_jobs=-1, verbose=1)

In [ ]:
#XGBoost with class imbalance handling
xgb_model = xgb.XGBClassifier(scale_pos_weight=len(y_train) / sum(y_train == 1), random_state=75)

In [ ]:
#Hyperparameter tuning the XGBoost model
param_grid_xgb = {
    'n_estimators': [10, 20, 30, 50],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5],
    'min_child_weight': [1, 3],
    'subsample': [0.8, 0.9],
    'colsample_bytree': [0.7],
    'gamma': [0, 0.1],
    'scale_pos_weight': [len(y_train) / sum(y_train == 1)],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [0, 0.1, 1]
}

xgb_model = xgb.XGBClassifier(random_state=42, scale_pos_weight=len(y_train) / sum(y_train == 1))

# Setup GridSearchCV with 5-fold cross-validation
grid_search_xgb = GridSearchCV(estimator=xgb_model, param_grid=param_grid_xgb, cv=5, n_jobs=-1, verbose=1)

In [ ]:
# Combine them in a Voting Classifier (either soft or hard voting)
voting_clf = VotingClassifier(estimators=[
    ('lr', lr),
    ('log_reg', logreg),
    # ('grid_search_lr', grid_search_lr),
    ('rf', rf),
    ('grid_search_rf',grid_search_rf),
    ('xgb', xgb_model),
    # ('grid_search_xgb',grid_search_xgb)
], voting='soft')  # 'soft' voting uses predicted probabilities, 'hard' uses majority class


In [ ]:
pipe = Pipeline([
    ('percentile_capping', FunctionTransformer(percentile_capping, validate=False)),
    ('scaler', FunctionTransformer(scaling, validate=False)),
    ('feature_selection', FeatureSelection(threshold=0.01)),
    ('pca', PCA(n_components=0.95)),
    ('classifier', voting_clf)
])

In [ ]:
X_train_final = custom_imputation(X_train[new_vars])
X_test_final = custom_imputation(X_test[new_vars])

In [ ]:
pipe.fit(X_train_final, y_train)

In [ ]:
# Make predictions
y_pred_lr = pipe.predict(X_test_final)
# Get train accuracy
train_accuracy = pipe.score(X_train_final, y_train)
print(f"Train Accuracy: {train_accuracy:.4f}")

# Get test accuracy
test_accuracy = pipe.score(X_test_final, y_test)
print(f"Test Accuracy: {test_accuracy:.4f}")

# Evaluate model performance
print("Classification Report:")
print(classification_report(y_test, y_pred_lr))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

# 7. Creating submission file

For submission, we need to make sure that the format is exactly the same as the sample.csv file. It contains 2 columns, id and churn_probability

In [ ]:
submission_data = unseen.set_index('id')[new_vars]

In [ ]:
submission_data_final = custom_imputation(submission_data)

In [ ]:
unseen['churn_probability'] = pipe.predict(submission_data_final)
output = unseen[['id','churn_probability']]
output.head()

In [ ]:
output.to_csv('SriChandanaG_GauriGujarathi.csv',index=False)